# Session 02: Date and Time with Python and Pandas

Welcome! In this session, you'll learn how to work with dates and times in Python and pandas, with a focus on practical business analytics and data science applications.

## Table of Contents
1. [Introduction: Why Dates Matter in Analytics](#introduction)
2. [Python's Built-in Date and Time Tools](#python-datetime)
    - Native `datetime` and `dateutil`
    - Parsing and formatting dates
3. [Pandas for Time Series and Business Data](#pandas-time-series)
    - Timestamps, Periods, Timedeltas
    - Creating and converting date columns
4. [Indexing and Slicing Time Series Data](#indexing)
5. [Generating Date Ranges for Business Use](#date-ranges)
6. [Practical Frequency Codes for Business](#frequency-codes)
7. [Advanced: Resampling, Shifting, and Rolling Windows](#advanced)

<a id="introduction"></a>

## Why Dates and Times Matter in Analytics

- **Sales trends**: How do sales change over time?
- **Customer behavior**: When do users interact with your product?
- **Forecasting**: Predicting future values based on time
- **Operations**: Scheduling, deadlines, and reporting

Let's get started!

<a id="python-datetime"></a>

## Python's Built-in Date and Time Tools

Python provides several ways to work with dates and times. Understanding these is essential for cleaning, transforming, and analyzing time-based data in business and data science projects.

The Python world has a number of available representations of dates, times, deltas, and timespans.
While the time series tools provided by Pandas tend to be the most useful for data science applications, it is helpful to see their relationship to other packages used in Python.

### Native Python dates and times: `datetime` and `dateutil`

Python's built-in `datetime` module and the third-party `dateutil` module are the foundation for handling dates and times. These are useful for parsing, formatting, and basic calculations—skills you'll use in any analytics or data science role.

#### Example: Recording a transaction in July 2025

In [1]:
from datetime import datetime

# Record a transaction on July 20, 2025 at 15:45
transaction_time = datetime(2025, 7, 20, 15, 45)
transaction_time

datetime.datetime(2025, 7, 20, 15, 45)

In [2]:
type(transaction_time)

datetime.datetime

#### Parsing dates from strings (real-world business data)

In business, dates often come as strings in various formats. `dateutil` can parse almost any date string you encounter in analytics projects.

Or, using the ``dateutil`` module, you can parse dates from a variety of string formats:

In [3]:
from dateutil import parser

# Parse different business date formats for July 2025
order_date1 = parser.parse("July 20, 2025")
order_date2 = parser.parse("20/07/2025")
order_date3 = parser.parse("2025-07-20")
order_date1, order_date2, order_date3

(datetime.datetime(2025, 7, 20, 0, 0),
 datetime.datetime(2025, 7, 20, 0, 0),
 datetime.datetime(2025, 7, 20, 0, 0))

### Formatting dates for business reports: `strftime`

The `strftime` method lets you create custom date strings for reports, dashboards, and presentations. This is essential for clear communication in analytics and data science.

This method states for "string from time" and it's very useful to transform a `datetime` variable into a formatted string according to the date and time format we want. All possible options here: https://docs.python.org/3/library/datetime.html#strftime-and-strptime-format-codes

In [4]:
from datetime import datetime

# Assuming transaction_time is a datetime object
transaction_time = datetime.now()
print(transaction_time)

# Format the transaction date for a business report
print(transaction_time.strftime("%A, %d %B %Y, %H:%M"))
print(transaction_time.strftime("Today: %d %B, %H:%M"))

2026-01-13 19:12:21.436542
Tuesday, 13 January 2026, 19:12
Today: 13 January, 19:12


In [5]:
transaction_time.strftime("%V")  # iso week

'03'

In [6]:
transaction_time.strftime("%Y-%m")

'2026-01'

In [7]:
transaction_time.strftime("%d-%B-%Y")

'13-January-2026'

<a id="pandas-time-series"></a>

## Pandas for Time Series and Business Data

Pandas is the go-to library for time series analysis in analytics and data science. It provides powerful tools for working with timestamps, periods, and timedeltas—essential for business forecasting, reporting, and operations.

This section will introduce the fundamental Pandas data structures for working with time series data:

- For *time stamps*, Pandas provides the ``Timestamp`` type. As mentioned before, it is essentially a replacement for Python's native ``datetime``, but is based on the more efficient ``numpy.datetime64`` data type. The associated Index structure is ``DatetimeIndex``.
- For *time Periods*, Pandas provides the ``Period`` type. This encodes a fixed-frequency interval based on ``numpy.datetime64``. The associated index structure is ``PeriodIndex``.
- For *time deltas* or *durations*, Pandas provides the ``Timedelta`` type. ``Timedelta`` is a more efficient replacement for Python's native ``datetime.timedelta`` type, and is based on ``numpy.timedelta64``. The associated index structure is ``TimedeltaIndex``.

**So, in general, for date and time manipulation in pandas bear in mind `Timestamp`, `Period` and `Timedelta`**

In [8]:
import numpy as np
import pandas as pd

# Set a reference year for reproducibility in examples
year = 2026

### Operating with `Timestamp` and `Period`

One of the useful things we can do with datetimes in pandas is checking if a specific timestamp is comprised inside a specific period

In [9]:
datetime.now()

datetime.datetime(2026, 1, 13, 19, 12, 21, 959890)

In [10]:
# Create a period for the current year (2025)
year_period = pd.Period(1765)
year_period.start_time, year_period.end_time

(Timestamp('1765-01-01 00:00:00'), Timestamp('1765-12-31 23:59:59.999999999'))

In [11]:
print(year_period.start_time)

1765-01-01 00:00:00


In [12]:
q3_2025 = pd.Period("2025Q3")

q3_2025.end_time

Timestamp('2025-09-30 23:59:59.999999999')

In [13]:
event_time = pd.Timestamp("2025-07-20 10:00")

q3_2025.start_time <= event_time <= q3_2025.end_time

True

### Creating datetimes with `pd.to_datetime` and `pd.to_timedelta` functions

This function tries to convert the provided input into a sequence of pandas datetime objects. The most common use of this function is to convets a **formatted string** into a **datetime**

In [14]:
dt_s = pd.Series(["1/1/26", "2/1/26", "3/1/26"])

dt_s

0    1/1/26
1    2/1/26
2    3/1/26
dtype: object

In [15]:
dt_s_converted = pd.to_datetime(dt_s)
dt_s_converted.dtype

/var/folders/m6/c304cbwn6016v9v4lv618s740000gn/T/ipykernel_96102/242336566.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt_s_converted = pd.to_datetime(dt_s)


dtype('<M8[ns]')

Or 

In [16]:
dt_s = pd.Series(["2026-01-01", "2026-01-02"])
dt_st = pd.Series(["1/1/26", "2/1/26"])

pd.to_datetime(dt_st)  # will work as expected

/var/folders/m6/c304cbwn6016v9v4lv618s740000gn/T/ipykernel_96102/1572684977.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(dt_st)  # will work as expected


0   2026-01-01
1   2026-02-01
dtype: datetime64[ns]

In [17]:
# Mixed-format July 2026 dates
dt_st = pd.Series(["2026-07-04", "07/20/2026", "July 31, 2026"])
pd.to_datetime(dt_st)  # will not work due to mixed formats

ValueError: time data "07/20/2026" doesn't match format "%Y-%m-%d", at position 1. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [ ]:
s = pd.Series(["01-07-2025", "31-07-2025"])
pd.to_datetime(s, dayfirst=True)  # dayfirst=True to handle European date format

0   2025-07-01
1   2025-07-31
dtype: datetime64[ns]

In [ ]:
pd.to_datetime(
    s, format="mixed"
)  # if we have mixed formats, we can specify 'mixed' to handle them

0   2026-07-04
1   2026-07-20
2   2026-07-31
dtype: datetime64[ns]

In [ ]:
# Convert a string date in July 2025 to pandas datetime
date = pd.to_datetime("20th of July, 2025, 4:25PM")
date

Timestamp('2025-07-20 16:25:00')

The detection of the format is done automatically, but sometimes it fails. For more saftey, we can provide directly the _format_ with the **format** argument. Only valid if the format is always the same

In [ ]:
date_format = "%d/%m/%Y"
date_format_2 = "%d of %B %Y, %HPM"

# use the "format" argument to provide the datetime format
dates = pd.to_datetime(
    ["2 of December 2026, 3PM", "3 of December 2026, 4PM"], format=date_format_2
)

print(dates)

# Use the "format" argument for consistent July 2026 dates
dates = pd.to_datetime(["20/07/2026", "31/07/2026"], format=date_format)

print(dates)

DatetimeIndex(['2026-12-02 03:00:00', '2026-12-03 04:00:00'], dtype='datetime64[ns]', freq=None)
DatetimeIndex(['2026-07-20', '2026-07-31'], dtype='datetime64[ns]', freq=None)


We can convert timestamps to periods using the `to_period` method. For example, to convert a timestamp to a hourly period:

In [ ]:
s = pd.Series(["2026-07-04", "07/20/2026", "July 31, 2026"])

pd.to_datetime(s, format="mixed")

0   2026-07-04
1   2026-07-20
2   2026-07-31
dtype: datetime64[ns]

In [ ]:
# Convert July 2025 dates to period (hourly)
print(dates)
dates.to_period("h").end_time

DatetimeIndex(['2026-07-20', '2026-07-31'], dtype='datetime64[ns]', freq=None)


DatetimeIndex(['2026-07-20 00:59:59.999999999', '2026-07-31 00:59:59.999999999'], dtype='datetime64[ns]', freq=None)

In that case, we will get the first hour of the day as the period if we specify 'H'.

Additionally, we can create timedeltas (time span) with the following code

In [ ]:
# Create a timedelta of 2.5 hours
span = pd.to_timedelta(2.5, unit="h")
span

Timedelta('0 days 02:30:00')

Timedeltas can be used to perform operations with datetime objects in pandas. For example:

In [ ]:
pd.to_datetime("21th of July, 2025 11:14") + span

Timestamp('2025-07-21 13:44:00')

In [ ]:
pd.to_datetime("2025-07-31 16:44") - span

Timestamp('2025-07-31 14:14:00')

The same with timedelta arrays

In [ ]:
list(np.arange(12))  # range(0, 12)

[np.int64(0),
 np.int64(1),
 np.int64(2),
 np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6),
 np.int64(7),
 np.int64(8),
 np.int64(9),
 np.int64(10),
 np.int64(11)]

In [ ]:
# Create a range of timedeltas (e.g., hourly shifts in July 2025)
spans = pd.to_timedelta(np.arange(12), "h")
spans

TimedeltaIndex(['0 days 00:00:00', '0 days 01:00:00', '0 days 02:00:00',
                '0 days 03:00:00', '0 days 04:00:00', '0 days 05:00:00',
                '0 days 06:00:00', '0 days 07:00:00', '0 days 08:00:00',
                '0 days 09:00:00', '0 days 10:00:00', '0 days 11:00:00'],
               dtype='timedelta64[ns]', freq=None)

In [ ]:
datetimes = pd.to_datetime("21th of July, 2025") + spans[3:6]

datetimes

DatetimeIndex(['2025-07-21 03:00:00', '2025-07-21 04:00:00',
               '2025-07-21 05:00:00'],
              dtype='datetime64[ns]', freq=None)

### Indexing by Time

Where the Pandas time series tools really become useful is when you begin to *index data by timestamps*.
For example, we can construct a ``Series`` object that has time indexed data:

In [ ]:
index = pd.DatetimeIndex(["2025-07-01", "2025-08-08", "2024-07-15", "2024-07-22"])

data = pd.Series([100, 200, 150, 300], index=index)
data

2025-07-01    100
2025-08-08    200
2024-07-15    150
2024-07-22    300
dtype: int64

There are additional special date-only indexing operations, such as passing a year to obtain a slice of all data from that year:

In [ ]:
data["2025"]

2025-07-01    100
2025-08-08    200
dtype: int64

In [ ]:
data["2025-07"]  # Filter July 2025

2025-07-01    100
dtype: int64

### Create sequences with `pd.date_range()`, `pd.period_range()` and `pd.timedelta_range()`

To make the creation of regular date sequences more convenient, Pandas offers a few functions for this purpose: ``pd.date_range()`` for timestamps, ``pd.period_range()`` for periods, and ``pd.timedelta_range()`` for time deltas.

In [ ]:
# Create a daily date range for July 2025
pd.date_range("2025-07-01", "2025-08-01", freq="h")

DatetimeIndex(['2025-07-01 00:00:00', '2025-07-01 01:00:00',
               '2025-07-01 02:00:00', '2025-07-01 03:00:00',
               '2025-07-01 04:00:00', '2025-07-01 05:00:00',
               '2025-07-01 06:00:00', '2025-07-01 07:00:00',
               '2025-07-01 08:00:00', '2025-07-01 09:00:00',
               ...
               '2025-07-31 15:00:00', '2025-07-31 16:00:00',
               '2025-07-31 17:00:00', '2025-07-31 18:00:00',
               '2025-07-31 19:00:00', '2025-07-31 20:00:00',
               '2025-07-31 21:00:00', '2025-07-31 22:00:00',
               '2025-07-31 23:00:00', '2025-08-01 00:00:00'],
              dtype='datetime64[ns]', length=745, freq='h')

In [ ]:
# Create an hourly date range for a week in July 2025
pd.date_range("2025-07-01", "2025-07-07", freq="h")

DatetimeIndex(['2025-07-01 00:00:00', '2025-07-01 01:00:00',
               '2025-07-01 02:00:00', '2025-07-01 03:00:00',
               '2025-07-01 04:00:00', '2025-07-01 05:00:00',
               '2025-07-01 06:00:00', '2025-07-01 07:00:00',
               '2025-07-01 08:00:00', '2025-07-01 09:00:00',
               ...
               '2025-07-06 15:00:00', '2025-07-06 16:00:00',
               '2025-07-06 17:00:00', '2025-07-06 18:00:00',
               '2025-07-06 19:00:00', '2025-07-06 20:00:00',
               '2025-07-06 21:00:00', '2025-07-06 22:00:00',
               '2025-07-06 23:00:00', '2025-07-07 00:00:00'],
              dtype='datetime64[ns]', length=145, freq='h')

Alternatively, the date range can be specified not with a start and endpoint, but with a startpoint and a number of periods:

In [ ]:
pd.date_range("2025-07-01 01:32", periods=8, freq="h")

DatetimeIndex(['2025-07-01 01:32:00', '2025-07-01 02:32:00',
               '2025-07-01 03:32:00', '2025-07-01 04:32:00',
               '2025-07-01 05:32:00', '2025-07-01 06:32:00',
               '2025-07-01 07:32:00', '2025-07-01 08:32:00'],
              dtype='datetime64[ns]', freq='h')

The spacing can be modified by altering the ``freq`` argument, which defaults to ``D``.
For example, here we will construct a range of hourly timestamps:

In [ ]:
# Create a date range with a custom frequency (every 2 days) in July 2025
pd.date_range("2025-07-01", periods=8, freq="2D")

DatetimeIndex(['2025-07-01', '2025-07-03', '2025-07-05', '2025-07-07',
               '2025-07-09', '2025-07-11', '2025-07-13', '2025-07-15'],
              dtype='datetime64[ns]', freq='2D')

To create regular sequences of ``Period`` or ``Timedelta`` values, the very similar ``pd.period_range()`` and ``pd.timedelta_range()`` functions are useful.
Here are some monthly periods:

In [ ]:
pd.period_range("2025-07", periods=2)

PeriodIndex(['2025-07-01', '2025-07-02'], dtype='period[D]')

And a sequence of durations increasing by an hour:

In [ ]:
# Create a timedelta range for business hours in July 2025
pd.timedelta_range(0, periods=10, freq="2h43min")

TimedeltaIndex(['0 days 00:00:00', '0 days 02:43:00', '0 days 05:26:00',
                '0 days 08:09:00', '0 days 10:52:00', '0 days 13:35:00',
                '0 days 16:18:00', '0 days 19:01:00', '0 days 21:44:00',
                '1 days 00:27:00'],
               dtype='timedelta64[ns]', freq='163min')

All of these require an understanding of Pandas frequency codes, which we'll summarize in the next section.

#### Frequencies and Offsets

Fundamental to these Pandas time series tools is the concept of a frequency or date offset.
Just as we saw the ``D`` (day) and ``H`` (hour) codes above, we can use such codes to specify any desired frequency spacing.
The following table summarizes the main codes available:

| Code   | Description         | Code   | Description          |
|--------|---------------------|--------|----------------------|
| ``D``  | Calendar day        | ``B``  | Business day         |
| ``W``  | Weekly              |        |                      |
| ``M``  | Month end           | ``BM`` | Business month end   |
| ``Q``  | Quarter end         | ``BQ`` | Business quarter end |
| ``A``  | Year end            | ``BA`` | Business year end    |
| ``h``  | Hours               | ``BH`` | Business hours       |
| ``min``  | Minutes             |        |                      |
| ``s``  | Seconds             |        |                      |
| ``L``  | Milliseonds         |        |                      |
| ``U``  | Microseconds        |        |                      |
| ``N``  | nanoseconds         |        |                      |

The monthly, quarterly, and annual frequencies are all marked at the end of the specified period.
By adding an ``S`` suffix to any of these, they instead will be marked at the beginning:

| Code    | Description            | Code    | Description            |
|---------|------------------------|---------|------------------------|
| ``MS``  | Month start            |``BMS``  | Business month start   |
| ``QS``  | Quarter start          |``BQS``  | Business quarter start |
| ``AS``  | Year start             |``BAS``  | Business year start    |

Additionally, you can change the month used to mark any quarterly or annual code by adding a three-letter month code as a suffix:

- ``Q-JAN``, ``BQ-FEB``, ``QS-MAR``, ``BQS-APR``, etc.
- ``A-JAN``, ``BA-FEB``, ``AS-MAR``, ``BAS-APR``, etc.

In the same way, the split-point of the weekly frequency can be modified by adding a three-letter weekday code:

- ``W-SUN``, ``W-MON``, ``W-TUE``, ``W-WED``, etc.

On top of this, codes can be combined with numbers to specify other frequencies.
For example, for a frequency of 2 hours 30 minutes, we can combine the hour (``H``) and minute (``T``) codes as follows:

In [ ]:
# Create a timedelta range with a custom frequency (every 2.5 hours)
pd.timedelta_range(0, periods=9, freq="2h17min")

TimedeltaIndex(['0 days 00:00:00', '0 days 02:17:00', '0 days 04:34:00',
                '0 days 06:51:00', '0 days 09:08:00', '0 days 11:25:00',
                '0 days 13:42:00', '0 days 15:59:00', '0 days 18:16:00'],
               dtype='timedelta64[ns]', freq='137min')

## Resampling, Shifting, and Windowing

The ability to use dates and times as indices to intuitively organize and access data is an important piece of the Pandas time series tools.
The benefits of indexed data in general (automatic alignment during operations, intuitive data slicing and access, etc.) still apply, and Pandas provides several additional time series-specific operations.

We will take a look at a few of those here, using some stock price data as an example, with Google.

In [18]:
import yfinance as yf

goog = yf.download("GOOG", start="2025-06-01", end="2026-01-12")
goog.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,GOOG,GOOG,GOOG,GOOG,GOOG
Date,,,,,
2025-06-02,169.902649,170.592752,168.187366,168.546379,24742900
2025-06-03,167.249954,169.334217,166.222766,168.401785,25386700
2025-06-04,168.925339,169.114820,167.334713,167.818383,18508700
2025-06-05,169.344177,171.887185,168.885448,171.149210,25375400
2025-06-06,174.440170,175.347678,171.827362,171.827362,22258100


For simplicity, we'll use just the closing price:

In [19]:
data = goog["Close"]

data

Ticker,GOOG
Date,
2025-06-02,169.902649
2025-06-03,167.249954
2025-06-04,168.925339
2025-06-05,169.344177
2025-06-06,174.440170
...,...
2026-01-05,317.320007
2026-01-06,314.549988
2026-01-07,322.429993


In [21]:
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_white"

In [22]:
px.line(data, title="GOOG Stock")


### Resampling and converting frequencies

One common need for time series data is resampling at a higher or lower frequency.
This can be done using the ``resample()`` method, or the much simpler ``asfreq()`` method.
The primary difference between the two is that ``resample()`` is fundamentally a *data aggregation*, while ``asfreq()`` is fundamentally a *data selection*.

Taking a look at the Google closing price, let's compare what the two return when we down-sample the data.
Here we will resample the data at the end of business year:

In [ ]:
data.head()

Date
2025-06-02    169.902649
2025-06-03    167.249954
2025-06-04    168.925339
2025-06-05    169.344193
2025-06-06    174.440170
Name: GOOG, dtype: float64

In [24]:
data_resample = data.resample("ME").mean()
data_freq = data.asfreq("ME")

In [25]:
data_resample

Ticker,GOOG
Date,
2025-06-30,173.365179
2025-07-31,185.627865
2025-08-31,202.562385
2025-09-30,242.757642
2025-10-31,254.738580
2025-11-30,293.151894
2025-12-31,313.475991
2026-01-31,320.795003


In [26]:
data_freq

Ticker,GOOG
Date,
2025-06-30,177.116043
2025-07-31,192.562134
2025-08-31,NaN
2025-09-30,243.391205
2025-10-31,281.636261
2025-11-30,NaN
2025-12-31,313.799988


In [27]:
# Create a single DataFrame for plotting using merged data
# using pd.merge

comparison_df = pd.merge(data_resample, data_freq, left_index=True, right_index=True)

# Create the line plot
figure = px.line(
    comparison_df, title="GOOG Stock (Monthly) - Resample vs AsFreq Comparison"
)

# Show the plot
figure.show()

In this case we've made a down-sampling of timeseries data

For up-sampling, ``resample()`` and ``asfreq()`` are largely equivalent, though resample has many more options available.
In this case, the default for both methods is to leave the up-sampled points empty, that is, filled with NA values.
Just as with the ``pd.fillna()`` function discussed previously, ``asfreq()`` accepts a ``method`` argument to specify how values are imputed.
Here, we will resample the business day data at a daily frequency (i.e., including weekends):

In [29]:
goog_d = data.asfreq("D")
goog_d_fill = data.asfreq("D", method="bfill")

goog_d.values

array([[169.90264893],
       [167.24995422],
       [168.92533875],
       [169.34417725],
       [174.44017029],
       [         nan],
       [         nan],
       [177.35566711],
       [179.73197937],
       [178.51387024],
       [176.69668579],
       [175.60836792],
       [         nan],
       [         nan],
       [177.66517639],
       [176.95628357],
       [173.71128845],
       [         nan],
       [167.47094727],
       [         nan],
       [         nan],
       [165.75361633],
       [167.48094177],
       [171.22515869],
       [174.16059875],
       [177.99467468],
       [         nan],
       [         nan],
       [177.11604309],
       [176.63677979],
       [179.4823761 ],
       [180.27116394],
       [         nan],
       [         nan],
       [         nan],
       [177.28578186],
       [174.88948059],
       [177.38562012],
       [178.42401123],
       [181.02996826],
       [         nan],
       [         nan],
       [182.52764893],
       [182

In [32]:
# Create proper DataFrame for plotting
# We use .flatten() to ensure we have 1D arrays, avoiding issues if yfinance returned a DataFrame
daily_df = pd.DataFrame({
    "empty_weekends": goog_d.values.flatten() + 2,
    "filled_weekends": goog_d_fill.values.flatten()
}, index=goog_d.index)

fig = px.line(
    daily_df,
    title="GOOG Stock (Daily)",
)
fig.show()

### Time-shifts with `shift()`

Another common time series-specific operation is shifting of data in time. The method is `shift()`

In [33]:
import pandas as pd

ts = pd.DataFrame({"timestamp": ["1", "2", "3", "4"], "value": [10, 20, 30, 35]})

ts["lag_1"] = ts["value"].shift(1)
ts["diff_1"] = ts["value"].diff(2)
ts["lag_minus1"] = ts["value"].shift(-1)

ts

,timestamp,value,lag_1,diff_1,lag_minus1
0,1,10,NaN,NaN,20.0
1,2,20,10.0,NaN,30.0
2,3,30,20.0,20.0,35.0
3,4,35,30.0,15.0,NaN


In [37]:
goog_sh = goog['Close'].shift(1)
goog_close = goog['Close']

daily_df = pd.DataFrame({
    "empty_weekends": goog_close.values.flatten() + 10,
    "filled_weekends": goog_sh.values.flatten()
}, index=goog.index)

daily_df.corr()

,empty_weekends,filled_weekends
empty_weekends,1.000000,0.996666
filled_weekends,0.996666,1.000000


### Rolling windows

Rolling statistics are a third type of time series-specific operation implemented by Pandas.
These can be accomplished via the ``rolling()`` attribute of ``Series`` and ``DataFrame`` objects, which returns a view similar to what we saw with the ``groupby`` operation
This rolling view makes available a number of aggregation operations by default.

For example, here is the one-year rolling mean and standard deviation of the Google stock prices:

In [ ]:
goog

Price,Close,High,Low,Open,Volume
Ticker,GOOG,GOOG,GOOG,GOOG,GOOG
Date,,,,,
2025-06-02,169.902649,170.592752,168.187366,168.546379,24742900
2025-06-03,167.249954,169.334217,166.222766,168.401785,25386700
2025-06-04,168.925339,169.114820,167.334713,167.818383,18508700
2025-06-05,169.344193,171.887201,168.885463,171.149225,25375400
2025-06-06,174.440170,175.347678,171.827362,171.827362,22258100
...,...,...,...,...,...
2026-01-05,317.320007,319.250000,315.248993,317.695007,19934000
2026-01-06,314.549988,321.559998,312.339996,317.309998,18989900


In [46]:
df = pd.DataFrame({'s': [1, 2, 3, 4]})

df['s_ma2'] = df['s'].rolling(3).mean()
df

,s,s_ma2
0,1,NaN
1,2,NaN
2,3,2.0
3,4,3.0


In [47]:
close_rolling_30_mean = goog['Close'].rolling(window=30).mean()
close = goog['Close']

In [48]:
rolling_df = pd.DataFrame({
    "close_rolling_mean_30": close_rolling_30_mean.values.flatten(),
    "close": close.values.flatten()
}, index=close.index)

fig = px.line(
    rolling_df,
    title="GOOG Stock (Daily) vs Rolling Mean (30 days)",
)
fig.show()

## The `dt` attribute in Series

The `dt` attribute of a pandas Series represents the datetime values of the series as a DatetimeIndex, which provides a lot of convenient functions for working with dates and times.

The `dt` attribute is only available for Series objects that contain datetime values. If the series does not contain datetime values, attempting to access the dt attribute will raise an `AttributeError`.

In [49]:
today = pd.to_datetime("2026-09-05")

In [51]:
today.day_of_week

5

In [ ]:
s = pd.Series(["2025-01-01", "2025-02-01", "2025-03-01"])

pd.to_datetime(s).dt.day_of_week

0   2025-01-01
1   2025-02-01
2   2025-03-01
dtype: datetime64[ns]

In [60]:
# Create a df with datetime values
s = pd.Series(["2025-01-01", "2025-02-01", "2025-03-01"])

df = pd.DataFrame(pd.to_datetime(s))

df

,0
0,2025-01-01
1,2025-02-01
2,2025-03-01


In [62]:
df["day_of_week"] = df[0].map(lambda x: x.day_of_week)
df

,0,day_of_week
0,2025-01-01,2
1,2025-02-01,5
2,2025-03-01,5


The `dt` attribute provides access to the following properties:

- `year`: the year of the datetime
- `month`: the month of the datetime
- `day`: the day of the datetime
- `hour`: the hour of the datetime
- `minute`: the minute of the datetime
- `second`: the second of the datetime

In [ ]:
# Get the year of each datetime
df[0].dt.year

0    2025
1    2025
2    2025
Name: 0, dtype: int32

In [ ]:
# Get the month of each datetime
df[0].dt.month

0    1
1    2
2    3
Name: 0, dtype: int32

In [ ]:
# Get the day of each datetime
df[0].dt.day

0    1
1    1
2    1
Name: 0, dtype: int32

In [ ]:
df = pd.DataFrame(
    {
        "date": pd.to_datetime(["2022-01-01", "2022-02-01", "2022-03-01"]),
        "values": [12, 23, 435],
    }
)

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["day_of_week"] = df["date"].dt.day_of_week

df.drop("date", axis=1)

,values,year,month,day,day_of_week
0,12,2022,1,1,5
1,23,2022,2,1,1
2,435,2022,3,1,1
